# 📖 Notebook 3: Surge Pricing

When everyone wants a ride at the same time (concert ending, rainstorm, New Year's Eve), there aren't enough drivers. Uber's solution: **raise the price** until some riders decide to wait and more drivers are incentivized to come online.

This is surge pricing — a dynamic multiplier applied to the base fare based on the supply/demand ratio in a geographic zone.

## Learning Objectives

By the end of this notebook, you'll understand:
- How to calculate supply (available drivers) and demand (ride requests) per zone
- How to compute a surge multiplier from the supply/demand ratio
- How zone-based pricing works with PostGIS spatial queries
- How to apply surge to fare estimates

## 🛠️ Setup

Start the infrastructure first:

```bash
cd 06-system-designs/uber
docker compose up -d
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import redis
import time
import random
import json

DB_CONFIG = {
    "host": "localhost", "port": 5432,
    "database": "uber_demo", "user": "demo", "password": "demo"
}
REDIS_CONFIG = {"host": "localhost", "port": 6379, "decode_responses": True}

def get_db():
    return psycopg2.connect(**DB_CONFIG)

def get_redis():
    return redis.Redis(**REDIS_CONFIG)

try:
    conn = get_db(); conn.close()
    print("✅ Connected to PostgreSQL + PostGIS")
except Exception as e:
    print(f"❌ PostgreSQL failed: {e}")

try:
    r = get_redis(); r.ping()
    print("✅ Connected to Redis")
except Exception as e:
    print(f"❌ Redis failed: {e}")

## 🤔 Why Surge Pricing?

Imagine a Friday night in downtown San Francisco:

- **50 people** open the app wanting a ride
- **10 drivers** are available in the area

Without surge: 10 riders get matched, 40 wait forever. Drivers have no incentive to come to the area.

With surge (2.5×): 
- Some riders decide it's too expensive and take the bus → demand drops to 25
- Drivers in nearby areas see the surge and drive to downtown → supply rises to 20
- More riders get served, drivers earn more, the market balances

**Surge pricing is a real-time market mechanism.** The multiplier goes up when demand > supply and comes back down when they balance.

In [ ]:
# Let's look at our surge zones

conn = get_db()
cur = conn.cursor()

cur.execute("""
    SELECT 
        id,
        zone_name,
        ST_Y(center::geometry) AS latitude,
        ST_X(center::geometry) AS longitude,
        radius_km,
        current_demand,
        current_supply,
        surge_multiplier
    FROM surge_zones
    ORDER BY id;
""")

print("📊 Surge Zones in San Francisco:")
print(f"{'ID':<4} {'Zone':<22} {'Lat':>8} {'Lng':>10} {'Radius':>7} {'Demand':>7} {'Supply':>7} {'Surge':>6}")
print("-" * 78)
for row in cur.fetchall():
    print(f"{row[0]:<4} {row[1]:<22} {row[2]:>8.4f} {row[3]:>10.4f} {row[4]:>5.1f}km {row[5]:>7} {row[6]:>7} {row[7]:>5.2f}×")

conn.close()
print()
print("💡 All zones start with demand=0, supply=0, surge=1.0×")
print("   We'll simulate traffic and watch surge change.")

## 🧱 A World Without Surge (the bad baseline)

Before we compute dynamic multipliers, let's see what happens with **fixed prices** when demand spikes.

Imagine a concert ends and 80 riders open the app in a zone where only 3 drivers are available.
With fixed prices:

- Price does not change, so **drivers have no extra incentive** to drive into the zone.
- The 3 drivers serve ~3 rides. The other 77 wait, retry, and eventually give up.
- Riders who *really* need a ride have no way to signal it.

Surge pricing is the market's way of solving this: raise the price, and two things happen at once —
some riders defer the trip (**demand drops**) and more drivers come to the zone (**supply rises**).

In [ ]:
# Fixed-price vs surge: same demand timeline, two pricing rules.
# We track every waiting rider's AGE in windows, so "gave up" means what it says
# instead of "the queue hit an arbitrary cap".

PRICE_WINDOWS = [
    ("T+0 concert ends",   80, 3),   # (label, new_requests_this_window, drivers_in_zone)
    ("T+2 still flooding", 70, 3),
    ("T+4",                50, 4),   # w/o surge: almost no extra drivers arrive
    ("T+6",                30, 4),
    ("T+8",                15, 4),
    ("T+10",                5, 4),
]

MAX_WAIT_WINDOWS = 2   # a rider who has waited longer than this closes the app


def simulate(pricing_rule):
    """Run the timeline under one pricing rule.

    Returns (served, gave_up, still_waiting).
    """
    waiting = []                       # [age_in_windows, rider_count], oldest first
    served_total = cancelled = 0

    print(f"{'Window':<22} {'New':>4} {'Drivers':>8} {'Served':>7} {'Gave up':>8} {'Backlog':>8}")
    for label, new, drivers in PRICE_WINDOWS:
        backlog = sum(count for _, count in waiting)

        # The pricing rule can only react to the backlog it can already see, so
        # surge always lags reality by one window. That lag is real, not a shortcut.
        adj_new, adj_drivers = pricing_rule(new, drivers, backlog)

        # Everyone already waiting ages by one window; this window's arrivals are fresh
        waiting = [[age + 1, count] for age, count in waiting]
        waiting.append([0, adj_new])

        # Drivers pick up the longest-waiting riders first
        capacity, served = adj_drivers, 0
        for cohort in waiting:
            take = min(cohort[1], capacity)
            cohort[1] -= take
            capacity -= take
            served += take
            if capacity == 0:
                break

        # Anyone who has now waited longer than MAX_WAIT_WINDOWS walks away
        gave_up = sum(count for age, count in waiting if age > MAX_WAIT_WINDOWS)
        waiting = [[age, count] for age, count in waiting
                   if age <= MAX_WAIT_WINDOWS and count > 0]

        served_total += served
        cancelled += gave_up
        print(f"{label:<22} {adj_new:>4} {adj_drivers:>8} {served:>7} {gave_up:>8} "
              f"{sum(c for _, c in waiting):>8}")

    left = sum(count for _, count in waiting)
    print(f"  → served {served_total}, gave up {cancelled}, still waiting {left}\n")
    return served_total, cancelled, left


def fixed_price(new, drivers, backlog):
    # Nothing changes — price is flat, nobody reacts
    return new, drivers


def surge_price(new, drivers, backlog):
    # High backlog raises the price → some riders defer, extra drivers come in
    if backlog > 20:
        return int(new * 0.6), drivers + 4    # surge kicks in
    if backlog > 10:
        return int(new * 0.8), drivers + 2
    return new, drivers


print("❌ Fixed price (no surge):")
fixed_served, fixed_lost, fixed_left = simulate(fixed_price)
print("✅ With surge pricing:")
surge_served, surge_lost, surge_left = simulate(surge_price)

print("💡 Same demand curve, different outcomes.")
print(f"   Rides served:   {fixed_served:>4} → {surge_served}")
print(f"   Riders lost:    {fixed_lost:>4} → {surge_lost}")
print(f"   Still waiting:  {fixed_left:>4} → {surge_left}")
print("   Surge is not about charging more — it's about clearing the market")
print("   so riders who need a ride now can actually get one.")

# If surge ever stops beating the fixed price on this timeline, this whole
# section is arguing for something it cannot demonstrate.
assert surge_served > fixed_served, (
    f"surge should clear more rides than a flat price: {surge_served} vs {fixed_served}"
)
assert surge_lost < fixed_lost, (
    f"surge should lose fewer riders to abandonment: {surge_lost} vs {fixed_lost}"
)
assert surge_left < fixed_left, (
    f"surge should leave a smaller backlog behind: {surge_left} vs {fixed_left}"
)

## 📊 Calculating the Surge Multiplier

The core formula is simple:

```
demand_supply_ratio = demand / supply

If ratio <= 1.0  → surge = 1.0× (no surge — enough drivers)
If ratio  > 1.0  → surge grows with the ratio, capped at some max
```

Real Uber uses much more sophisticated models, but the core idea is the same: **price rises when demand outpaces supply**.

In [ ]:
def calculate_surge(demand, supply, max_surge=5.0):
    """
    Calculate the surge multiplier based on demand and supply.
    
    Args:
        demand: Number of ride requests in the zone
        supply: Number of available drivers in the zone
        max_surge: Maximum surge multiplier (cap)
    
    Returns:
        Surge multiplier (1.0 = no surge)
    """
    if supply == 0:
        # No drivers at all — max surge to attract drivers
        return max_surge if demand > 0 else 1.0
    
    ratio = demand / supply
    
    if ratio <= 1.0:
        return 1.0  # enough drivers, no surge
    
    # Surge grows with the square root of the ratio
    # This makes surge increase quickly at first, then slow down
    # Example: ratio=2 → 1.50×, ratio=4 → 2.00×, ratio=9 → 3.00×
    # (sqrt(2) is 1.41, but the 0.25 rounding step below lifts it to 1.50 —
    #  quote the number the function actually returns, not the one before rounding)
    surge = ratio ** 0.5
    
    # Round to nearest 0.25 (Uber-style pricing increments)
    surge = round(surge * 4) / 4
    
    return min(surge, max_surge)


# Show how surge changes with different demand/supply ratios
print("📊 Surge Multiplier Examples:")
print(f"{'Demand':>7} {'Supply':>7} {'Ratio':>7} {'Surge':>7}")
print("-" * 32)

examples = [
    (5, 10),    # more drivers than requests
    (10, 10),   # balanced
    (15, 10),   # slight shortage
    (20, 10),   # 2:1 ratio
    (40, 10),   # 4:1 ratio
    (90, 10),   # 9:1 ratio (concert ending)
    (50, 0),    # no drivers at all
]

for demand, supply in examples:
    ratio = demand / supply if supply > 0 else "∞"
    surge = calculate_surge(demand, supply)
    ratio_str = f"{ratio:.1f}" if isinstance(ratio, float) else ratio
    print(f"{demand:>7} {supply:>7} {ratio_str:>7} {surge:>6.2f}×")

# The docstring promises specific numbers. Hold the function to them.
assert calculate_surge(5, 10) == 1.0, "surplus supply must never surge"
assert calculate_surge(10, 10) == 1.0, "a balanced zone must never surge"
assert calculate_surge(20, 10) == 1.5, f"ratio 2 → 1.5×, got {calculate_surge(20, 10)}"
assert calculate_surge(40, 10) == 2.0, f"ratio 4 → 2.0×, got {calculate_surge(40, 10)}"
assert calculate_surge(90, 10) == 3.0, f"ratio 9 → 3.0×, got {calculate_surge(90, 10)}"
assert calculate_surge(1000, 10) == 5.0, "surge must be capped at max_surge"
assert calculate_surge(50, 0) == 5.0, "zero supply with demand → max surge"
assert calculate_surge(0, 0) == 1.0, "zero supply with zero demand → no surge"

## 🗺️ Zone-Based Supply & Demand Counting

To calculate surge per zone, we need to count:
1. **Supply**: How many available drivers are within each zone's radius
2. **Demand**: How many ride requests originated from each zone recently

For supply, we use PostGIS to count drivers within each zone's geographic boundary.  
For demand, we count recent ride requests (tracked in Redis for speed).

In [ ]:
# First, let's load driver locations into Redis for this demo

r = get_redis()
conn = get_db()
cur = conn.cursor()

r.delete("drivers:locations", "drivers:available")

cur.execute("""
    SELECT d.id, d.status,
           ST_X(dl.location::geometry) AS lng,
           ST_Y(dl.location::geometry) AS lat
    FROM drivers d
    JOIN driver_locations dl ON d.id = dl.driver_id;
""")
for row in cur.fetchall():
    did, status, lng, lat = row
    r.geoadd("drivers:locations", (lng, lat, f"driver:{did}"))
    if status == "available":
        r.sadd("drivers:available", f"driver:{did}")

conn.close()
print(f"✅ Loaded {r.zcard('drivers:locations')} drivers into Redis")
print(f"   {r.scard('drivers:available')} are available")

In [ ]:
def count_supply_in_zone(zone_lng, zone_lat, radius_km):
    """Count available drivers within a zone using Redis Geo."""
    r = get_redis()
    nearby = r.geosearch(
        name="drivers:locations",
        longitude=zone_lng, latitude=zone_lat,
        radius=radius_km, unit="km"
    )
    available = r.smembers("drivers:available")
    return len([d for d in nearby if d in available])


def simulate_demand(zone_id, count):
    """Simulate ride requests in a zone by incrementing a Redis counter."""
    r = get_redis()
    # In production, this would be incremented each time a ride is requested
    # The counter resets every pricing window (e.g., every 2 minutes)
    r.set(f"zone:{zone_id}:demand", count, ex=120)  # expires in 2 min


def get_demand(zone_id):
    """Get the current demand count for a zone."""
    r = get_redis()
    val = r.get(f"zone:{zone_id}:demand")
    return int(val) if val else 0


# Simulate different demand levels across zones
# Downtown: Friday night rush. SoMa: convention ending. Others: normal.
demand_simulation = {
    1: 35,  # Downtown/Financial — happy hour
    2: 50,  # SoMa/Convention — big event ending
    3: 8,   # Mission — normal evening
    4: 5,   # Castro — quiet
    5: 12,  # Marina — moderate
    6: 25,  # SFO Airport — flight arrivals
}

for zone_id, demand in demand_simulation.items():
    simulate_demand(zone_id, demand)

print("✅ Simulated demand across 6 zones")

In [ ]:
# Now calculate surge for each zone!

conn = get_db()
cur = conn.cursor()

cur.execute("""
    SELECT id, zone_name,
           ST_X(center::geometry) AS lng,
           ST_Y(center::geometry) AS lat,
           radius_km
    FROM surge_zones ORDER BY id;
""")
zones = cur.fetchall()

zone_demand, zone_supply, zone_surge, zone_ratio = {}, {}, {}, {}

print("🔥 Surge Pricing Calculation:")
print(f"{'Zone':<22} {'Supply':>7} {'Demand':>7} {'Ratio':>7} {'Surge':>7} {'Status':>12}")
print("-" * 68)

for zone_id, name, lng, lat, radius_km in zones:
    supply = count_supply_in_zone(lng, lat, float(radius_km))
    demand = get_demand(zone_id)
    surge = calculate_surge(demand, supply)
    ratio = f"{demand/supply:.1f}" if supply > 0 else "∞"
    
    if surge >= 3.0:
        status = "🔴 EXTREME"
    elif surge >= 2.0:
        status = "🟠 HIGH"
    elif surge > 1.0:
        status = "🟡 ACTIVE"
    else:
        status = "🟢 NORMAL"
    
    zone_demand[zone_id] = demand
    zone_supply[zone_id] = supply
    zone_surge[zone_id] = surge
    # No drivers at all is an infinite shortage -- unless nobody wants a ride,
    # in which case there is no shortage to price.
    zone_ratio[zone_id] = (demand / supply if supply > 0
                           else (float("inf") if demand > 0 else 0.0))

    print(f"{name:<22} {supply:>7} {demand:>7} {ratio:>7} {surge:>6.2f}× {status:>12}")
    
    # Update the surge in the database
    cur.execute("""
        UPDATE surge_zones 
        SET current_demand = %s, current_supply = %s, 
            surge_multiplier = %s, updated_at = NOW()
        WHERE id = %s;
    """, (demand, supply, surge, zone_id))

conn.commit()
conn.close()

print()
print("💡 In production, this calculation runs every 1-2 minutes per zone.")
print("   Zones with more demand than supply get a surge multiplier.")

# --- guardrails -----------------------------------------------------------
# The demand counters carry a 120s TTL. If you came back from lunch mid-notebook
# they are gone, every zone reads 0, and the rest of the notebook quietly prices
# everything at 1.0× while looking perfectly healthy. Fail loudly instead.
assert any(d > 0 for d in zone_demand.values()), (
    "every zone read demand 0 — the zone:*:demand counters have a 120s TTL and have "
    "expired. Re-run the demand-simulation cell above, then this one."
)

# Contract of calculate_surge, checked against live data rather than a table:
for zid, surge in zone_surge.items():
    d, sup = zone_demand[zid], zone_supply[zid]
    assert 1.0 <= surge <= 5.0, f"zone {zid}: surge {surge} outside [1.0, 5.0]"
    if sup > 0 and d <= sup:
        assert surge == 1.0, f"zone {zid}: demand {d} <= supply {sup} but surge is {surge}×"

# Surge must be monotone in the demand/supply ratio. If a zone with a worse
# shortage prices lower than a zone with a milder one, the formula is broken.
ranked = sorted(zone_ratio.items(), key=lambda kv: kv[1])
for (a, ra), (b, rb) in zip(ranked, ranked[1:]):
    assert zone_surge[a] <= zone_surge[b], (
        f"surge must rise with the shortage: zone {a} (ratio {ra:.2f}) prices "
        f"{zone_surge[a]}× but zone {b} (ratio {rb:.2f}) prices {zone_surge[b]}×"
    )

assert max(zone_surge.values()) > 1.0, (
    "no zone surged at all — the demand simulation did nothing useful"
)

## 💰 Applying Surge to Fare Estimates

When a rider requests a fare estimate, the system:
1. Calculates the base fare from distance and time
2. Determines which surge zone the pickup is in
3. Multiplies the base fare by the zone's surge multiplier

One thing that is *not* optional here: **do the money in integer cents.**

`40.13 * 5.0` is `200.64999999999998` in binary floating point. Round that and you get a
number a cent away from what the rider gets when they multiply the two figures printed on
their own receipt. And if you apply the multiplier to the *unrounded* base while displaying
the *rounded* one, the receipt stops adding up entirely — a support ticket per ride.

So: round to cents exactly once, then stay in integers. Surge is always a multiple of 0.25,
so `surge * 4` is an exact integer and `(cents * quarters + 2) // 4` is exact half-up
rounding with no float in sight.

Let's build this end-to-end.

In [ ]:
def estimate_fare(pickup_lng, pickup_lat, dropoff_lng, dropoff_lat):
    """
    Calculate a fare estimate with surge pricing.
    
    Returns a dict with base fare, surge multiplier, and final estimate.
    """
    conn = get_db()
    cur = conn.cursor()
    
    # Step 1: Calculate distance between pickup and dropoff (in km)
    cur.execute("""
        SELECT ST_Distance(
            ST_SetSRID(ST_MakePoint(%s, %s), 4326)::geography,
            ST_SetSRID(ST_MakePoint(%s, %s), 4326)::geography
        ) / 1000.0 AS distance_km;
    """, (pickup_lng, pickup_lat, dropoff_lng, dropoff_lat))
    distance_km = float(cur.fetchone()[0])
    
    # Step 2: Calculate the base fare -- and immediately pin it to integer CENTS.
    # Simple pricing: $2.50 base + $1.50/km + $0.25/min (assume 2 min/km in city)
    base_charge = 2.50
    per_km = 1.50
    per_min = 0.25
    estimated_minutes = distance_km * 2  # rough estimate: 2 min/km in city
    base_cents = round(
        (base_charge + (distance_km * per_km) + (estimated_minutes * per_min)) * 100
    )
    
    # Step 3: Find the surge zone containing the pickup location
    cur.execute("""
        SELECT zone_name, surge_multiplier
        FROM surge_zones
        WHERE ST_DWithin(
            center,
            ST_SetSRID(ST_MakePoint(%s, %s), 4326)::geography,
            radius_km * 1000  -- convert km to meters
        )
        ORDER BY ST_Distance(
            center,
            ST_SetSRID(ST_MakePoint(%s, %s), 4326)::geography
        )
        LIMIT 1;
    """, (pickup_lng, pickup_lat, pickup_lng, pickup_lat))
    
    zone_row = cur.fetchone()
    zone_name = zone_row[0] if zone_row else "No Zone"
    surge = float(zone_row[1]) if zone_row else 1.0
    
    # Step 4: Apply surge -- still in integer cents.
    #
    # The obvious `round(base_fare * surge, 2)` is wrong twice over. First it
    # multiplies the UNROUNDED base by the surge while the receipt shows the
    # rounded one, so the two printed numbers no longer produce the third.
    # Second, binary floats: 40.13 * 5.0 is 200.64999999999998, and rounding
    # that is a coin flip on the last cent.
    #
    # calculate_surge only ever emits multiples of 0.25, so surge * 4 is an exact
    # integer and (cents * quarters + 2) // 4 is exact half-up rounding.
    assert abs(surge * 4 - round(surge * 4)) < 1e-9, (
        f"surge {surge} is not a multiple of 0.25 -- the integer-cent maths below "
        "assumes the quarter-step pricing from calculate_surge"
    )
    surge_quarters = round(surge * 4)
    fare_cents = (base_cents * surge_quarters + 2) // 4
    
    conn.close()
    
    return {
        "distance_km": round(distance_km, 2),
        "estimated_minutes": round(estimated_minutes, 0),
        "base_fare": base_cents / 100,
        "surge_zone": zone_name,
        "surge_multiplier": surge,
        "estimated_fare": fare_cents / 100,
        # Keep the integers around: they are the source of truth, the dollars
        # above are just a rendering of them.
        "base_cents": base_cents,
        "fare_cents": fare_cents,
    }


# Test with different pickup locations
# All going to SFO Airport (-122.3790, 37.6213)
dropoff = (-122.3790, 37.6213)

pickups = [
    ("Downtown/Financial", -122.4000, 37.7900),
    ("SoMa (convention)",  -122.4000, 37.7800),
    ("Mission District",   -122.4200, 37.7600),
    ("Castro (thin supply)", -122.4350, 37.7600),
]

print("💰 Fare Estimates: Various Pickups → SFO Airport")
print("=" * 80)

estimates = {}
for label, plng, plat in pickups:
    fare = estimate_fare(plng, plat, *dropoff)
    estimates[label] = fare
    print(f"\n  📍 Pickup: {label}")
    print(f"     Distance:   {fare['distance_km']} km")
    print(f"     Base fare:  ${fare['base_fare']:.2f}")
    print(f"     Surge zone: {fare['surge_zone']} ({fare['surge_multiplier']}×)")
    print(f"     Final fare: ${fare['estimated_fare']:.2f}", end="")
    if fare['surge_multiplier'] > 1.0:
        extra = (fare['fare_cents'] - fare['base_cents']) / 100
        print(f"  (⚡ +${extra:.2f} surge)")
    else:
        print("  (no surge)")

surging = [lbl for lbl, f in estimates.items() if f["surge_multiplier"] > 1.0]
print()
print(f"⚠️  Honest caveat — {len(surging)} of {len(estimates)} pickups are surging right now.")
print("   These zones are 1–1.5 km circles holding a handful of drivers, so the")
print("   demand/supply ratio is computed from single-digit counts: one driver")
print("   going offline can swing the multiplier from 1.0× to 5.0×. Production")
print("   surge smooths over a rolling window and refuses to price off samples")
print("   this small. This lab does neither, deliberately, so the noise is visible.")

# The arithmetic must hold, and the pickup must resolve to a real zone (a silent
# "No Zone" would price at 1.0× and look like a working no-surge result).
for label, fare in estimates.items():
    assert fare["surge_zone"] != "No Zone", (
        f"{label}: pickup fell outside every surge zone — it would silently price at 1.0×"
    )
    assert fare["distance_km"] > 0, f"{label}: zero-length trip"

    # The dollar figures are only a rendering of the cents; they must round-trip.
    assert round(fare["base_fare"] * 100) == fare["base_cents"], (
        f"{label}: base ${fare['base_fare']} does not round-trip to {fare['base_cents']}c"
    )
    assert round(fare["estimated_fare"] * 100) == fare["fare_cents"], (
        f"{label}: fare ${fare['estimated_fare']} does not round-trip to {fare['fare_cents']}c"
    )

    # And the receipt has to add up: the fare the rider is quoted must be exactly
    # the displayed base times the displayed multiplier, half-up to the cent.
    expected_cents = (fare["base_cents"] * round(fare["surge_multiplier"] * 4) + 2) // 4
    assert fare["fare_cents"] == expected_cents, (
        f"{label}: ${fare['base_fare']:.2f} × {fare['surge_multiplier']} should quote "
        f"${expected_cents / 100:.2f}, got ${fare['estimated_fare']:.2f}"
    )

## 🔄 Watching Surge Change Over Time

Surge pricing isn't static — it recalculates every 1-2 minutes. Let's simulate demand changing and watch surge respond.

In [ ]:
# Simulate 3 pricing windows for the SoMa/Convention zone
# Scenario: A big tech conference just ended

zone_id = 2  # SoMa/Convention
zone_supply = 3  # 3 available drivers in this area

# Demand over time: spike then gradual decline
demand_over_time = [
    ("6:00 PM — Conference ends", 60),
    ("6:02 PM — Peak exodus",     80),
    ("6:04 PM — Surge attracts drivers", 50),
    ("6:06 PM — Demand falling",  30),
    ("6:08 PM — Some wait for Muni", 15),
    ("6:10 PM — Back to normal",  5),
]

# Assume supply increases as drivers respond to surge
supply_over_time = [3, 3, 5, 8, 10, 10]

print("📊 SoMa/Convention Zone — Surge Over Time")
print("=" * 70)
print(f"{'Time':<35} {'Demand':>7} {'Supply':>7} {'Surge':>7} {'Visual'}")
print("-" * 70)

surge_curve = []
for i, (label, demand) in enumerate(demand_over_time):
    supply = supply_over_time[i]
    surge = calculate_surge(demand, supply)
    surge_curve.append(surge)
    bar = "🔥" * int(surge)
    print(f"{label:<35} {demand:>7} {supply:>7} {surge:>6.2f}× {bar}")

print()
print("💡 Notice how:")
print("   1. Surge spikes when demand >> supply")
print("   2. High surge attracts more drivers (supply increases)")
print("   3. Higher supply + lower demand → surge comes back down")
print("   4. This creates a self-balancing market")

# Those four claims are testable. Test them.
peak = max(surge_curve)
peak_at = surge_curve.index(peak)
assert peak_at <= 1, (
    f"surge should spike in the first window or two, peaked at window {peak_at}: {surge_curve}"
)
assert all(a >= b for a, b in zip(surge_curve[peak_at:], surge_curve[peak_at + 1:])), (
    f"after the peak, surge must fall monotonically as supply catches up: {surge_curve}"
)
assert surge_curve[-1] == 1.0, (
    f"surge must settle back to 1.0× once supply exceeds demand, got {surge_curve[-1]}×"
)
assert supply_over_time[-1] > supply_over_time[0], (
    "the scenario is supposed to show supply responding to surge"
)

## 🧹 Cleanup

In [ ]:
r = get_redis()
# Clean up demand counters and driver data
for key in r.keys("zone:*:demand"):
    r.delete(key)
r.delete("drivers:locations", "drivers:available")

# Reset surge zones to defaults
conn = get_db()
cur = conn.cursor()
cur.execute("""
    UPDATE surge_zones 
    SET current_demand = 0, current_supply = 0, surge_multiplier = 1.00;
""")
conn.commit()
conn.close()

print("🧹 Cleaned up Redis keys and reset surge zones")

## 📚 Summary

### Key Takeaways

1. **Surge pricing balances supply and demand** — it's a real-time market mechanism, not just "charging more"
2. **The formula is simple**: `surge = f(demand / supply)` — capped at a maximum multiplier
3. **Zone-based pricing** uses PostGIS to determine which zone a pickup falls in
4. **Surge is self-correcting** — high prices attract more drivers AND reduce demand
5. **Recalculate frequently** — every 1-2 minutes per zone to respond to changing conditions
6. **Watch the sample size** — a 1 km zone holding two drivers prices off a two-sample
   estimate. One driver going offline doubles the ratio. Real systems smooth over a rolling
   window and floor the multiplier when counts are tiny; this lab does not, so you can see
   the jitter for yourself
7. **Surge always lags** — the multiplier is computed from the *previous* window's backlog,
   so it arrives one window late. That lag is visible in the fixed-vs-surge simulation:
   the first window is identical under both rules

### How This Fits in a System Design Interview

Surge pricing shows you understand:
- **Supply/demand economics** in distributed systems
- **Geographic partitioning** — different zones have independent pricing
- **Real-time aggregation** — counting requests per zone per time window
- **Combining Redis + PostGIS** — fast counters + spatial queries

### Next Up

In **Notebook 4**, we'll tackle the **trip lifecycle** — managing ride state from request to completion, with distributed locking to prevent double-assignment of drivers.